Chatbot and RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approcahes for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using Langsmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those dataset
3. How to measure your application's performance using different evaluation metrics

Overview

A typical RAG evaluation workflow consists of three main steps:

1. Creeating a dataset with questions and their expected answer
2. Running your RAG application on those questions
3. Using evaluation to measure how well your application performed, looking at factors like:

    . Answer relevance
    . Answer accuarcy 
    . Retrieval quality

For this tutorical, we'll create and evaluate a bot answer questions about a few of lilian Weeng insightful blog posts.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACKING"] = "true"


In [3]:
import truststore   
truststore.inject_into_ssl()

In [9]:
### create the datapoints
from langsmith import Client
client = Client()

## Define  the dataset - these are your test data
dataset_name = "Navdeep Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id, 
    examples=[{
        "inputs": {"question": "what is langchain?"},
        "outputs": {"answer": "A framework for building LLM applications"}
        },
        {
            "inputs": {"question": "what is langsmith?"},
            "outputs": {"answer": "A platform for observing and managing LLM applications"}
        },
        {
            "inputs": {"question": "what is OpenAI?"},
            "outputs": {"answer": "A platform for managing LLM applications"}
        },
        {
            "inputs": {"question": "what is Google?"},
            "outputs": {"answer": "A technology company specializing in search and other services"}
        },
        {
            "inputs": {"question": "what is Mistral?"},
            "outputs": {"answer": "A company developing open-weight language models"}
        },
    ]
)

{'example_ids': ['9116de1e-046f-4309-ba99-156291cce422',
  '2d032a63-5052-4200-a039-2cbd6975d145',
  'ce0754a3-0653-42bb-a527-3ff0ba94d3e8',
  'a170c192-200b-4adb-a276-9426780fc990',
  'efe0361e-2f21-49dc-b33a-d949de7982a6'],
 'count': 5,
 'as_of': '2026-10-23T05:48:50.717104646Z'}

### Define Metrics (LLM AS A Judge)

In [43]:
import openai
from langsmith import wrappers

openai_client = wrappers.wrap_openai(openai.OpenAI())

eval_instructions = "You are an expert professor specialized in grading students's answer to questions"

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following questions:
        {inputs['question']}"
        Here is the real answer:
        {reference_outputs['answer']}"
        You are grading the folliwng predicted answer:
        {outputs['response']}
        respond with CORRECT or INCORRECT:
        Grade:
    """
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "user", "content": eval_instructions},
            {"role": "user", "content": user_content}
        ]
    ).choices[0].message.content
    
    return response.upper() == "CORRECT"


In [30]:
## Concisions - checks whether the actual output is less than 2 x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return len(outputs['response']) < 2 * len(reference_outputs['answer'])   
    

### Run Evaludation

In [40]:
default_instructions = "Respond to the users question in a short,  concise manner (one short sentence)."

def my_app(question: str, model: str="gpt-4o-mini", instructions: str=default_instructions) -> str:
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ]
    )
    return response.choices[0].message.content



In [41]:
### Call my_app for every datapoints
def ls_target(inputs:str) -> dict:
    return {"response": my_app(inputs["question"])}


In [44]:
## Run our evaluation
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="openai-4o-mini-chatbot",
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-61c055fa' at:
https://smith.langchain.com/o/ad944d8b-e2b1-4fa4-a22d-8054b1844931/datasets/13193ea3-d860-41b4-bff1-f6be39aab819/compare?selectedSessions=2d2a0b2f-6bc4-4741-885c-124d6cb92b27




5it [00:10,  2.12s/it]
